# 📐 Notebook 02 — Konsep Dasar Statistika & Pembersihan Data

**Mata Kuliah**: Data Science  
**Tema**: E-Business  
**Dataset**: E-Commerce Shopee Indonesia (Kaggle)

---

## 1. Konsep Dasar Statistika

### 1.1 Definisi Statistika

**Statistika** adalah ilmu yang mempelajari cara mengumpulkan, mengorganisasi, menganalisis, menginterpretasi, dan mempresentasikan data. Dalam Data Science, statistika menjadi fondasi utama untuk memahami data secara kuantitatif.

Statistika terbagi dua:
- **Statistika Deskriptif**: Merangkum dan mendeskripsikan data yang ada (mean, median, modus, dll)
- **Statistika Inferensial**: Menarik kesimpulan tentang populasi berdasarkan sampel

### 1.2 Populasi vs Sampel

| Konsep | Definisi | Dalam Proyek Ini |
|---|---|---|
| **Populasi** | Keseluruhan objek/individu yang diteliti | Seluruh transaksi toko Shopee sepanjang masa |
| **Sampel** | Sebagian dari populasi yang dipilih untuk diteliti | Transaksi periode Des 2023 – Nov 2025 |
| **Parameter** | Ukuran yang mendeskripsikan populasi (μ, σ) | Rata-rata pembayaran seluruh pelanggan |
| **Statistik** | Ukuran yang mendeskripsikan sampel (x̄, s) | Rata-rata pembayaran dalam dataset ini |

### 1.3 Jenis Variabel

**Variabel Kuantitatif** (Numerik):
- **Diskrit**: Nilai bulat, dihitung (contoh: `total_qty`, `num_product_categories`)
- **Kontinu**: Nilai bisa pecahan (contoh: `Total Pembayaran`, `total_weight_gr`)

**Variabel Kualitatif** (Kategorikal):
- **Nominal**: Kategori tanpa urutan (contoh: `Metode Pembayaran`, `Provinsi`)
- **Ordinal**: Kategori dengan urutan (contoh: `Status Pesanan` — Selesai > Dikirim > Dibatalkan)

### 1.4 Skala Pengukuran

| Skala | Karakteristik | Contoh Kolom |
|---|---|---|
| **Nominal** | Kategorisasi saja, tidak ada urutan | `Metode Pembayaran`, `Provinsi`, `Opsi Pengiriman` |
| **Ordinal** | Ada urutan, jarak tidak pasti | `Status Pesanan` |
| **Interval** | Ada jarak pasti, tidak ada nol mutlak | — |
| **Rasio** | Ada nol mutlak, perbandingan bermakna | `Total Pembayaran`, `total_qty`, `Total Diskon` |

### 1.5 Ukuran Pemusatan Data

| Ukuran | Rumus | Keterangan |
|---|---|---|
| **Mean (Rata-rata)** | x̄ = Σxᵢ / n | Sensitif terhadap outlier |
| **Median** | Nilai tengah setelah diurutkan | Robust terhadap outlier |
| **Modus** | Nilai yang paling sering muncul | Untuk data kategorikal |

### 1.6 Ukuran Penyebaran Data

| Ukuran | Rumus | Keterangan |
|---|---|---|
| **Range** | Xmax − Xmin | Sederhana, sensitif outlier |
| **Varians** | s² = Σ(xᵢ−x̄)² / (n−1) | Kuadrat satuan asli |
| **Std Dev** | s = √varians | Satuan sama dengan data asli |
| **IQR** | Q3 − Q1 | Robust, untuk deteksi outlier |

---
## 2. Setup & Load Data

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from utils import set_style, save_fig
set_style()

# Load data dari hasil notebook 01
DATA_PATH = '../data/processed/all_months_clean.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'✅ Data loaded | Shape: {df.shape}')
df.head(3)

---
## 3. Audit Kualitas Data (Sebelum Cleaning)

In [ ]:
print('=== AUDIT KUALITAS DATA ===')
print(f'Total baris     : {len(df):,}')
print(f'Total kolom     : {len(df.columns)}')
print(f'Data duplikat   : {df.duplicated().sum():,}')
print()

# Missing values per kolom
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct
}).query('`Jumlah Missing` > 0').sort_values('Persentase (%)', ascending=False)

print('=== MISSING VALUES ===')
if len(missing_df) == 0:
    print('✅ Tidak ada missing values!')
else:
    print(missing_df.to_string())

In [ ]:
# Visualisasi missing values
if df.isnull().sum().sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    missing_pct[missing_pct > 0].sort_values(ascending=True).plot(
        kind='barh', ax=ax, color='#e76f51'
    )
    ax.set_title('Persentase Missing Values per Kolom', fontweight='bold')
    ax.set_xlabel('Persentase (%)')
    plt.tight_layout()
    plt.savefig('../reports/figures/02_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('📊 Tidak ada missing values — visualisasi dilewati')

---
## 4. Data Cleaning

In [ ]:
df_clean = df.copy()
print(f'Shape awal: {df_clean.shape}')

# ── 4.1 Hapus duplikat
before = len(df_clean)
df_clean.drop_duplicates(subset='order_id', keep='first', inplace=True)
after = len(df_clean)
print(f'\n[1] Hapus duplikat berdasarkan order_id: {before - after:,} baris dihapus')

# ── 4.2 Konversi tipe data datetime
df_clean['Waktu Pesanan Dibuat'] = pd.to_datetime(
    df_clean['Waktu Pesanan Dibuat'], errors='coerce'
)
print(f'[2] Konversi datetime: {df_clean["Waktu Pesanan Dibuat"].isna().sum()} error')

# ── 4.3 Konversi kolom numerik
num_cols = ['total_qty', 'total_weight_gr', 'Total Diskon',
            'Ongkos Kirim Dibayar oleh Pembeli',
            'Estimasi Potongan Biaya Pengiriman',
            'Total Pembayaran', 'Perkiraan Ongkos Kirim']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
print(f'[3] Konversi numerik: {len(num_cols)} kolom dikonversi')

# ── 4.4 Normalisasi teks Status Pesanan
df_clean['Status Pesanan'] = df_clean['Status Pesanan'].str.strip()
df_clean.loc[
    df_clean['Status Pesanan'].str.contains('Pesanan diterima', na=False),
    'Status Pesanan'
] = 'Pesanan Diterima'
df_clean['Status Pesanan'] = df_clean['Status Pesanan'].replace(
    {'Telah Dikirim': 'Sedang Dikirim'}
)
print(f'[4] Normalisasi Status Pesanan: selesai')

# ── 4.5 Buat fitur turunan
df_clean['Tahun'] = df_clean['Waktu Pesanan Dibuat'].dt.year
df_clean['Bulan'] = df_clean['Waktu Pesanan Dibuat'].dt.month
df_clean['Nama_Bulan'] = df_clean['Waktu Pesanan Dibuat'].dt.strftime('%Y-%m')
df_clean['Hari_Minggu'] = df_clean['Waktu Pesanan Dibuat'].dt.day_name()
df_clean['Jam'] = df_clean['Waktu Pesanan Dibuat'].dt.hour
print(f'[5] Fitur turunan waktu: Tahun, Bulan, Nama_Bulan, Hari_Minggu, Jam')

# ── 4.6 Isi missing values kolom Alasan Pembatalan
df_clean['Alasan Pembatalan'] = df_clean['Alasan Pembatalan'].fillna('Tidak Dibatalkan')
print(f'[6] Isi missing Alasan Pembatalan: selesai')

print(f'\nShape akhir: {df_clean.shape}')

### 4.1 Klasifikasi Variabel dalam Dataset

In [ ]:
# Klasifikasi variabel sesuai teori statistika
variabel_info = {
    'order_id': ('Kualitatif-Nominal', 'ID Unik', 'Identifier'),
    'total_qty': ('Kuantitatif-Diskrit', 'Rasio', 'Jumlah item pesanan'),
    'total_weight_gr': ('Kuantitatif-Diskrit', 'Rasio', 'Berat total (gram)'),
    'total_returned_qty': ('Kuantitatif-Diskrit', 'Rasio', 'Jumlah item retur'),
    'Total Diskon': ('Kuantitatif-Kontinu', 'Rasio', 'Nilai diskon (Rp)'),
    'product_categories': ('Kualitatif-Nominal', 'Nominal', 'Kategori produk'),
    'num_product_categories': ('Kuantitatif-Diskrit', 'Rasio', 'Jumlah kategori berbeda'),
    'Status Pesanan': ('Kualitatif-Ordinal', 'Ordinal', 'Status akhir pesanan'),
    'Opsi Pengiriman': ('Kualitatif-Nominal', 'Nominal', 'Layanan pengiriman'),
    'Metode Pembayaran': ('Kualitatif-Nominal', 'Nominal', 'Cara pembayaran'),
    'Kota/Kabupaten': ('Kualitatif-Nominal', 'Nominal', 'Kota tujuan'),
    'Provinsi': ('Kualitatif-Nominal', 'Nominal', 'Provinsi tujuan'),
    'Ongkos Kirim Dibayar oleh Pembeli': ('Kuantitatif-Kontinu', 'Rasio', 'Biaya kirim (Rp)'),
    'Estimasi Potongan Biaya Pengiriman': ('Kuantitatif-Kontinu', 'Rasio', 'Subsidi ongkir (Rp)'),
    'Total Pembayaran': ('Kuantitatif-Kontinu', 'Rasio', 'Nilai transaksi total (Rp)'),
    'Perkiraan Ongkos Kirim': ('Kuantitatif-Kontinu', 'Rasio', 'Estimasi ongkir (Rp)'),
    'Waktu Pesanan Dibuat': ('Kuantitatif-Kontinu', 'Interval', 'Timestamp pesanan'),
}

var_df = pd.DataFrame(
    [(k, v[0], v[1], v[2]) for k, v in variabel_info.items()],
    columns=['Kolom', 'Jenis Variabel', 'Skala Pengukuran', 'Keterangan']
)
var_df

---
## 5. Simpan Data Bersih

In [ ]:
# Simpan data yang sudah dibersihkan
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/all_months_cleaned.csv', index=False)
print('✅ Data bersih disimpan ke: data/processed/all_months_cleaned.csv')
print(f'   Total baris: {len(df_clean):,}')
print(f'   Total kolom: {len(df_clean.columns)}')
print(f'\n=== RINGKASAN PROSES CLEANING ===')
print(f'Baris sebelum cleaning : {len(df):,}')
print(f'Baris setelah cleaning : {len(df_clean):,}')
print(f'Baris dihapus          : {len(df) - len(df_clean):,}')
print(f'Kolom baru ditambahkan : Tahun, Bulan, Nama_Bulan, Hari_Minggu, Jam')

---
## 6. Ringkasan

Pada notebook ini telah dilakukan:

1. ✅ **Konsep statistika dasar**: populasi, sampel, jenis variabel, skala pengukuran, ukuran pemusatan & penyebaran
2. ✅ **Audit kualitas data**: identifikasi missing values dan duplikat
3. ✅ **Data cleaning**: hapus duplikat, konversi tipe data, normalisasi teks
4. ✅ **Feature engineering**: ekstraksi waktu (tahun, bulan, hari, jam)
5. ✅ **Klasifikasi variabel**: setiap kolom diklasifikasikan sesuai teori statistika

➡️ Lanjut ke: `03_exploratory_analysis.ipynb`